# Task 2: Customer Preference Analysis

## Objective

Analyze how **cuisine** relates to ratings and popularity in the restaurant dataset.

Cognifyz requires three analyses:

1. Relationship between cuisine type and restaurant rating
2. Most popular cuisines based on **number of votes**
3. Whether particular cuisines tend to receive higher **observed** ratings

This notebook documents how multi-label `Cuisines` values are split, how votes are attributed, and why small samples are not treated as broadly representative.


## Dataset / Input

- `../data/engineered_dataset.csv` (Level 2 Task 3 engineered file)
- Original `Cuisines` strings are preserved; they are exploded only on a working copy


## Imports


In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 120

BASE_DIR = Path.cwd()
DATA_PATH = (BASE_DIR / "../data/engineered_dataset.csv").resolve()
FIG_DIR = BASE_DIR / "outputs" / "figures"
TAB_DIR = BASE_DIR / "outputs" / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

MIN_RESTAURANTS = 30
TOP_N = 15

print("Working directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Minimum restaurants for cuisine-level claims:", MIN_RESTAURANTS)


Working directory: C:\Users\adity\Documents\Cognifyz\Level-3\Task-2_Customer_Preference_Analysis
Dataset path: C:\Users\adity\Documents\Cognifyz\Level-3\data\engineered_dataset.csv
Minimum restaurants for cuisine-level claims: 30


## Load Dataset


In [2]:
df = pd.read_csv(DATA_PATH)
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print("Cuisines missing:", int(df["Cuisines"].isna().sum()))
print("Cuisines empty string:", int((df["Cuisines"].fillna("").str.strip() == "").sum()))
df[["Restaurant ID", "Cuisines", "Aggregate rating", "Votes"]].head(10)


Rows: 9551 Columns: 26
Cuisines missing: 9
Cuisines empty string: 9


,Restaurant ID,Cuisines,Aggregate rating,Votes
0,6317637,"French, Japanese, Desserts",4.8,314
1,6304287,Japanese,4.5,591
2,6300002,"Seafood, Asian, Filipino, Indian",4.4,270
3,6318506,"Japanese, Sushi",4.9,365
4,6314302,"Japanese, Korean",4.8,229
5,18189371,Chinese,4.4,336
6,6300781,"Asian, European",4.0,520
7,6301290,"Seafood, Filipino, Asian, European",4.2,677
8,6300010,"European, Asian, Indian",4.9,621
9,6314987,Filipino,4.8,532


## Data Validation — Cuisine Field Structure

`Cuisines` is a comma-separated multi-label field. Treating `"Indian, Chinese"` as a single atomic category would undercount both Indian and Chinese restaurants.

**Transformation**

1. Drop rows with missing/blank `Cuisines` from cuisine-level tables (9 rows).
2. Split on commas and strip whitespace.
3. Explode so each (restaurant, cuisine) pair is one row.

**Vote attribution (documented choice)**

Each listed cuisine receives the restaurant’s **full** `Votes` and `Aggregate rating`.

Rationale: Cognifyz defines popularity as votes associated with the cuisine. A diner voting for a North Indian + Chinese restaurant is evidence of interest in both listed cuisines.

Consequence: votes of multi-cuisine restaurants are counted in more than one cuisine. Totals across cuisines therefore exceed the dataset-wide vote sum. This is association, not a partition of votes.


In [3]:
print("Sample raw Cuisines values:")
print(df["Cuisines"].dropna().head(12).tolist())
print()
print("Restaurants with multiple cuisine labels:", int(df["Cuisines"].dropna().str.contains(",").sum()))
print("Unique raw Cuisines strings:", df["Cuisines"].nunique(dropna=True))

cuisine_long = df.dropna(subset=["Cuisines"]).copy()
cuisine_long = cuisine_long[cuisine_long["Cuisines"].str.strip() != ""]
cuisine_long["Cuisine"] = cuisine_long["Cuisines"].str.split(",")
cuisine_long = cuisine_long.explode("Cuisine")
cuisine_long["Cuisine"] = cuisine_long["Cuisine"].str.strip()
cuisine_long = cuisine_long[cuisine_long["Cuisine"] != ""]
before_dedupe = len(cuisine_long)
cuisine_long = cuisine_long.drop_duplicates(subset=["Restaurant ID", "Cuisine"])

print()
print("Long-format rows before de-duplicating repeated labels:", before_dedupe)
print("Long-format rows (restaurant × cuisine):", len(cuisine_long))
print("Duplicate (restaurant, cuisine) pairs dropped:", before_dedupe - len(cuisine_long))
print("Distinct cuisine labels:", cuisine_long["Cuisine"].nunique())
print("Rows dropped for missing/blank Cuisines:", int(df["Cuisines"].isna().sum()))


Sample raw Cuisines values:
['French, Japanese, Desserts', 'Japanese', 'Seafood, Asian, Filipino, Indian', 'Japanese, Sushi', 'Japanese, Korean', 'Chinese', 'Asian, European', 'Seafood, Filipino, Asian, European', 'European, Asian, Indian', 'Filipino', 'Filipino, Mexican', 'American, Ice Cream, Desserts']

Restaurants with multiple cuisine labels: 6148
Unique raw Cuisines strings: 1825

Long-format rows before de-duplicating repeated labels: 19710
Long-format rows (restaurant × cuisine): 19705
Duplicate (restaurant, cuisine) pairs dropped: 5
Distinct cuisine labels: 145
Rows dropped for missing/blank Cuisines: 9


## Preparation — Cuisine-Level Metrics


In [4]:
cuisine_stats = (
    cuisine_long.groupby("Cuisine", as_index=False)
    .agg(
        n_restaurants=("Restaurant ID", "nunique"),
        n_rows=("Restaurant ID", "size"),
        total_votes=("Votes", "sum"),
        mean_votes=("Votes", "mean"),
        mean_rating=("Aggregate rating", "mean"),
        median_rating=("Aggregate rating", "median"),
        min_rating=("Aggregate rating", "min"),
        max_rating=("Aggregate rating", "max"),
    )
    .sort_values("n_restaurants", ascending=False)
    .reset_index(drop=True)
)

cuisine_stats["meets_min_n"] = cuisine_stats["n_restaurants"] >= MIN_RESTAURANTS
cuisine_stats.to_csv(TAB_DIR / "cuisine_stats_all.csv", index=False)

print("Cuisine labels:", len(cuisine_stats))
print(f"Labels with n_restaurants >= {MIN_RESTAURANTS}:", int(cuisine_stats["meets_min_n"].sum()))
print()
print("Top 10 by restaurant count:")
display(cuisine_stats.head(10))


Cuisine labels: 145
Labels with n_restaurants >= 30: 39

Top 10 by restaurant count:


,Cuisine,n_restaurants,n_rows,total_votes,mean_votes,mean_rating,median_rating,min_rating,max_rating,meets_min_n
0,North Indian,3960,3960,595981,150.500253,2.510455,3.1,0.0,4.9,True
1,Chinese,2733,2733,363879,133.142700,2.620234,3.1,0.0,4.9,True
2,Fast Food,1986,1986,184058,92.677744,2.563092,3.1,0.0,4.9,True
3,Mughlai,994,994,151944,152.861167,2.612575,3.1,0.0,4.9,True
4,Italian,764,764,329265,430.975131,3.562042,3.8,0.0,4.9,True
5,Bakery,744,744,57690,77.540323,2.440054,3.1,0.0,4.9,True
6,Continental,736,736,288255,391.650815,3.518750,3.7,0.0,4.9,True
7,Cafe,703,703,177568,252.586060,3.321195,3.7,0.0,4.9,True
8,Desserts,653,653,105889,162.157734,2.978867,3.4,0.0,4.9,True
9,South Indian,636,636,80962,127.298742,2.469811,3.0,0.0,4.9,True


## Analysis — Cuisine vs Rating

Average and median `Aggregate rating` by cuisine, restricted to labels with at least 30 restaurants so a handful of outlets cannot dominate the story.


In [5]:
eligible = cuisine_stats[cuisine_stats["meets_min_n"]].copy()
by_rating = eligible.sort_values("mean_rating", ascending=False).reset_index(drop=True)
by_rating.to_csv(TAB_DIR / "cuisine_rating_min30.csv", index=False)

print("Highest observed mean ratings (n >= 30):")
display(by_rating.head(TOP_N)[["Cuisine", "n_restaurants", "mean_rating", "median_rating", "total_votes"]])
print()
print("Lowest observed mean ratings (n >= 30):")
display(by_rating.tail(10)[["Cuisine", "n_restaurants", "mean_rating", "median_rating", "total_votes"]])


Highest observed mean ratings (n >= 30):


,Cuisine,n_restaurants,mean_rating,median_rating,total_votes
0,Sandwich,53,4.066038,4.1,23500
1,Steak,62,3.985484,4.0,25677
2,Sushi,75,3.973333,3.9,20582
3,Breakfast,41,3.965854,4.1,16097
4,Mediterranean,112,3.948214,4.0,80538
5,Bar Food,39,3.933333,4.1,15684
6,Indian,70,3.918571,4.1,22396
7,European,148,3.910811,4.0,103309
8,BBQ,33,3.903030,4.1,9647
9,Seafood,174,3.862069,4.0,74254



Lowest observed mean ratings (n >= 30):


,Cuisine,n_restaurants,mean_rating,median_rating,total_votes
29,Ice Cream,226,2.583186,3.20,15780
30,Fast Food,1986,2.563092,3.10,184058
31,North Indian,3960,2.510455,3.10,595981
32,South Indian,636,2.469811,3.00,80962
33,Bakery,744,2.440054,3.10,57690
34,Biryani,177,2.420339,3.10,18614
35,Street Food,562,2.356050,3.00,43798
36,Tibetan,44,2.302273,3.05,4593
37,Raw Meats,114,2.152632,2.90,1945
38,Mithai,380,2.073684,2.80,19647


## Analysis — Popularity by Votes

Popularity is **total associated votes** after full attribution, not restaurant count.


In [6]:
by_votes = cuisine_stats.sort_values("total_votes", ascending=False).reset_index(drop=True)
by_votes.to_csv(TAB_DIR / "cuisine_popularity_by_votes.csv", index=False)

print("Most popular cuisines by associated votes (all labels):")
display(by_votes.head(TOP_N)[["Cuisine", "n_restaurants", "total_votes", "mean_rating", "median_rating"]])

print()
print("Dataset-wide sum of Votes (each restaurant once):", int(df["Votes"].sum()))
print("Sum of cuisine-associated votes (full attribution):", int(cuisine_stats["total_votes"].sum()))
print("Ratio (attribution overlap):", round(cuisine_stats["total_votes"].sum() / df["Votes"].sum(), 3))


Most popular cuisines by associated votes (all labels):


,Cuisine,n_restaurants,total_votes,mean_rating,median_rating
0,North Indian,3960,595981,2.510455,3.1
1,Chinese,2733,363879,2.620234,3.1
2,Italian,764,329265,3.562042,3.8
3,Continental,736,288255,3.518750,3.7
4,Fast Food,1986,184058,2.563092,3.1
5,American,390,183117,3.661538,3.8
6,Cafe,703,177568,3.321195,3.7
7,Mughlai,994,151944,2.612575,3.1
8,Desserts,653,105889,2.978867,3.4
9,Asian,233,104303,3.815021,3.9



Dataset-wide sum of Votes (each restaurant once): 1498645
Sum of cuisine-associated votes (full attribution): 3914293
Ratio (attribution overlap): 2.612


## Visualizations


In [7]:
plot_votes = by_votes.head(TOP_N).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_votes["Cuisine"], plot_votes["total_votes"], color="steelblue")
ax.set_title(f"Top {TOP_N} Cuisines by Associated Votes")
ax.set_xlabel("Total associated votes (full attribution)")
ax.set_ylabel("Cuisine")
fig.tight_layout()
fig.savefig(FIG_DIR / "top_cuisines_by_votes.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_21932\4039297491.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
plot_n = cuisine_stats.head(TOP_N).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_n["Cuisine"], plot_n["n_restaurants"], color="teal")
ax.set_title(f"Top {TOP_N} Cuisines by Number of Restaurants")
ax.set_xlabel("Number of restaurants listing the cuisine")
ax.set_ylabel("Cuisine")
fig.tight_layout()
fig.savefig(FIG_DIR / "top_cuisines_by_restaurant_count.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_21932\4102782183.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
plot_r = by_rating.head(TOP_N).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_r["Cuisine"], plot_r["mean_rating"], color="darkorange")
ax.set_title(f"Highest Mean Aggregate Rating (cuisines with ≥ {MIN_RESTAURANTS} restaurants)")
ax.set_xlabel("Mean Aggregate rating")
ax.set_ylabel("Cuisine")
ax.set_xlim(0, 5)
fig.tight_layout()
fig.savefig(FIG_DIR / "top_cuisines_by_mean_rating.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_21932\1626527417.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
scatter = eligible.copy()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scatter["total_votes"], scatter["mean_rating"], s=np.clip(scatter["n_restaurants"] / 3, 20, 200), alpha=0.7)
for _, row in scatter.nlargest(8, "total_votes").iterrows():
    ax.annotate(row["Cuisine"], (row["total_votes"], row["mean_rating"]), fontsize=8, xytext=(5, 4), textcoords="offset points")
ax.set_title("Cuisine popularity vs observed mean rating (n ≥ 30)")
ax.set_xlabel("Total associated votes")
ax.set_ylabel("Mean Aggregate rating")
fig.tight_layout()
fig.savefig(FIG_DIR / "cuisine_votes_vs_mean_rating.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_21932\3743063385.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results


In [11]:
print("=== Popularity (votes) top 10 ===")
print(by_votes.head(10)[["Cuisine", "n_restaurants", "total_votes", "mean_rating"]].to_string(index=False))
print()
print("=== Highest mean rating (n >= 30) top 10 ===")
print(by_rating.head(10)[["Cuisine", "n_restaurants", "mean_rating", "median_rating", "total_votes"]].to_string(index=False))
print()
print("North Indian vs overall:")
overall_mean = float(df["Aggregate rating"].mean())
ni = cuisine_stats.loc[cuisine_stats["Cuisine"] == "North Indian"].iloc[0]
print("Overall restaurant mean rating:", round(overall_mean, 3))
print("North Indian mean rating:", round(float(ni["mean_rating"]), 3), "n=", int(ni["n_restaurants"]))


=== Popularity (votes) top 10 ===
     Cuisine  n_restaurants  total_votes  mean_rating
North Indian           3960       595981     2.510455
     Chinese           2733       363879     2.620234
     Italian            764       329265     3.562042
 Continental            736       288255     3.518750
   Fast Food           1986       184058     2.563092
    American            390       183117     3.661538
        Cafe            703       177568     3.321195
     Mughlai            994       151944     2.612575
    Desserts            653       105889     2.978867
       Asian            233       104303     3.815021

=== Highest mean rating (n >= 30) top 10 ===
      Cuisine  n_restaurants  mean_rating  median_rating  total_votes
     Sandwich             53     4.066038            4.1        23500
        Steak             62     3.985484            4.0        25677
        Sushi             75     3.973333            3.9        20582
    Breakfast             41     3.965854     

## Findings

The numbered answers below are filled from the computed tables in the Results cell (not from an external write-up).

1. **Most popular cuisines by votes** are the labels with the largest associated vote totals after exploding multi-cuisine restaurants. High restaurant-count cuisines (often North Indian, Chinese, Fast Food in this dataset) typically dominate votes because they appear on many menus, not necessarily because each outlet is more voted-on.
2. **Cuisine vs rating**: mean `Aggregate rating` varies by cuisine, but popular mass-market labels are not automatically the highest-rated. Sample size matters: a cuisine with 8 restaurants can look “excellent” by chance.
3. **Higher observed averages** are reported only for cuisines with at least 30 restaurants. These are **descriptive** differences in this table. They are not evidence that a cuisine *causes* higher ratings (location, price, selection bias, and who bothers to rate all confound the comparison).

Caveats:

- 9 restaurants have missing cuisines and are omitted from cuisine tables.
- Full vote attribution double-counts multi-label restaurants.
- The dataset is dominated by Indian cities, especially New Delhi, so “popular” largely reflects that market.
- Unrated restaurants (`Aggregate rating = 0`) pull some cuisine means downward.


## Limitations

- Exploding multi-label fields treats co-occurring cuisines as independent.
- No statistical hypothesis test is claimed as causal.
- Rare cuisines are excluded from ranking charts by design (`n >= 30`).


## Conclusion

After splitting comma-separated `Cuisines` and attributing each restaurant’s votes to every listed label, the dataset shows a small set of high-volume cuisines dominating popularity, while mean ratings differ more modestly once small samples are filtered out. All statements are about observed associations in this file.
